# EPLAN Electric P8 — Fine-Tune v2 with QLoRA

Fine-tunes **Qwen 2.5 3B Instruct** on **5,116 Q&A pairs** (v2 dataset) from EPLAN documentation.

**Changes from v1:**
- Dataset v2: +120 code pairs, +708 coverage pairs (was 4,288 → now 5,116)
- 2 epochs (was 3 — overfitting detected at step 120 in v1)
- Train/eval split 90/10 to monitor overfitting
- Eval + save every 40 steps

**Stack:** Transformers + TRL + PEFT + BitsAndBytes (native HuggingFace, no Unsloth)
**Runtime:** Kaggle GPU T4 x2 (free tier)
**Time:** ~1-2 hours

## 1. Install Dependencies

In [ ]:
%%capture install_log
import subprocess, sys
pip = [sys.executable, "-m", "pip", "install", "-U", "--quiet"]

# Upgrade numpy + scipy together to keep them compatible
subprocess.check_call(pip + ["numpy>=2.0", "scipy>=1.14"])

# Reinstall torchvision matching Kaggle's torch
subprocess.check_call(pip + ["torchvision"])

# Install HuggingFace stack
subprocess.check_call(pip + [
    "transformers>=4.51.0",
    "datasets>=3.3.0",
    "accelerate>=1.4.0",
    "bitsandbytes>=0.45.0",
    "trl>=0.16.0",
    "peft>=0.15.0",
    "sentencepiece",
])

# Restart runtime
import os
os._exit(0)

In [ ]:
# Verify installations after kernel restart
import numpy, torch, transformers, peft, trl, bitsandbytes
print(f"numpy:          {numpy.__version__}")
print(f"torch:          {torch.__version__}")
print(f"transformers:   {transformers.__version__}")
print(f"peft:           {peft.__version__}")
print(f"trl:            {trl.__version__}")
print(f"bitsandbytes:   {bitsandbytes.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU:            {torch.cuda.get_device_name() if torch.cuda.is_available() else 'N/A'}")

# Login to HuggingFace (needed for gated models like Gemma)
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
login(token=UserSecretsClient().get_secret("HF_TOKEN"))
print("HuggingFace: logged in")

## 2. Load Model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
MAX_SEQ_LENGTH = 2048

torch_dtype = torch.bfloat16

print(f"GPU 0: {torch.cuda.get_device_name(0)}")
print(f"GPU 1: {torch.cuda.get_device_name(1)}")
print(f"Precision: {torch_dtype}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch_dtype,
    device_map="balanced",
    attn_implementation="eager",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "right"

print(f"Model loaded: {MODEL_ID}")
print(f"Parameters: {model.num_parameters():,}")

## 3. Configure LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for QLoRA training (skip float32 upcast to save VRAM)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()

# LoRA config
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 4. Load & Format Dataset

In [ ]:
from datasets import load_dataset

# Load v2 dataset from HuggingFace Hub
dataset = load_dataset("covaga/eplan-qa-dataset", data_files="eplan_qa_v2_FINAL.jsonl", split="train")

# Train/eval split 90/10
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Dataset v2: {len(dataset)} total")
print(f"  Train: {len(train_dataset)}")
print(f"  Eval:  {len(eval_dataset)}")
print(f"Columns: {dataset.column_names}")
print(dataset[0])

In [ ]:
SYSTEM_MSG = """You are an expert EPLAN Electric P8 assistant specialized in industrial electrical engineering. 
You help engineers with API usage, troubleshooting, best practices, and procedural guidance. 
You can write complete, working C# scripts using the EPLAN API.
Provide accurate, detailed answers based on EPLAN documentation."""

def format_chat(example):
    """Convert instruction/output to Qwen chat format."""
    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]
    return {"messages": messages}

train_dataset = train_dataset.map(format_chat, remove_columns=["source", "category"])
eval_dataset = eval_dataset.map(format_chat, remove_columns=["source", "category"])

# Verify format
print("Sample formatted:")
sample_text = tokenizer.apply_chat_template(
    train_dataset[0]["messages"], tokenize=False, add_generation_prompt=False
)
print(sample_text[:600])

## 5. Train

In [ ]:
from trl import SFTConfig, SFTTrainer

# NOTE: eval during training causes OOM on T4.
# We evaluate manually after training instead.
training_args = SFTConfig(
    output_dir="eplan-qwen-qlora-v2",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    optim="adamw_8bit",
    logging_steps=10,
    save_strategy="steps",
    save_steps=40,
    learning_rate=2e-4,
    bf16=True,
    max_grad_norm=0.3,
    warmup_steps=10,
    lr_scheduler_type="constant",
    weight_decay=0.01,
    seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

print("Starting training v2 (no eval to avoid OOM)...")
stats = trainer.train()
print(f"\nTraining complete!")
print(f"  Loss: {stats.training_loss:.4f}")
print(f"  Runtime: {stats.metrics['train_runtime']:.0f}s")

trainer.save_model()

## 5b. Training Loss Curve

In [ ]:
import matplotlib.pyplot as plt
import json, glob

# Try trainer first, fallback to checkpoint logs
try:
    log_history = trainer.state.log_history
except NameError:
    # Read from latest checkpoint's trainer_state.json
    checkpoints = sorted(glob.glob("eplan-qwen-qlora-v2/checkpoint-*/trainer_state.json"))
    if checkpoints:
        with open(checkpoints[-1]) as f:
            log_history = json.load(f)["log_history"]
        print(f"Loaded logs from {checkpoints[-1]}")
    else:
        print("ERROR: No trainer or checkpoint found. Run training first.")
        raise SystemExit

train_steps = [x["step"] for x in log_history if "loss" in x]
train_losses = [x["loss"] for x in log_history if "loss" in x]

if not train_losses:
    print("No loss data found")
    raise SystemExit

# Epoch boundary (approx half of total steps)
epoch1_end = len(train_steps) // 2 * 10

plt.figure(figsize=(12, 5))
plt.plot(train_steps, train_losses, "b-", alpha=0.4, linewidth=1)
plt.plot(train_steps, train_losses, "b.", markersize=3)

# Moving average
window = 10
avg = [sum(train_losses[max(0,i-window):i+1])/len(train_losses[max(0,i-window):i+1]) for i in range(len(train_losses))]
plt.plot(train_steps, avg, "r-", linewidth=2, label="Moving avg (10)")

plt.axvline(x=epoch1_end, color="gray", linestyle="--", alpha=0.7, label="~Epoch 1 end")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training Loss — EPLAN v2 (Qwen 2.5 3B, QLoRA, 2 epochs)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("training_loss_v2.png", dpi=150)
plt.show()

print(f"Min: {min(train_losses):.4f} at step {train_steps[train_losses.index(min(train_losses))]}")
print(f"Final: {train_losses[-1]:.4f}")

## 6. Test the Model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from peft import PeftModel
import glob

torch.cuda.empty_cache()

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# Find latest checkpoint
checkpoints = sorted(glob.glob("eplan-qwen-qlora-v2/checkpoint-*"))
lora_path = checkpoints[-1] if checkpoints else "eplan-qwen-qlora-v2"
print(f"Loading LoRA from: {lora_path}")

# Load base model with 4-bit for inference
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config, device_map="balanced", attn_implementation="eager",
    )
except Exception as e:
    print(f"4-bit failed ({e}), loading in bfloat16...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16, device_map="balanced", attn_implementation="eager",
    )

model = PeftModel.from_pretrained(model, lora_path)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print("Model + LoRA loaded!")

SYSTEM_MSG = """You are an expert EPLAN Electric P8 assistant specialized in industrial electrical engineering. 
You help engineers with API usage, troubleshooting, best practices, and procedural guidance. 
You can write complete, working C# scripts using the EPLAN API.
Provide accurate, detailed answers based on EPLAN documentation."""

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

test_questions = [
    "How do I export a project to PDF using the EPLAN API?",
    "What is the difference between a Function and a FunctionBase?",
    "My script throws NullReferenceException when iterating pages. What could cause this?",
]

for q in test_questions:
    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user", "content": q},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    outputs = pipe(
        prompt, max_new_tokens=512, do_sample=True, temperature=0.7, top_p=0.9,
        eos_token_id=[tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|im_end|>")],
    )
    response = outputs[0]["generated_text"][len(prompt):].strip()
    print(f"\nQ: {q}")
    print(f"A: {response}")
    print("-" * 80)

## 7. Push to HuggingFace Hub

Already logged in from Step 1.

## 8. Merge LoRA + Push

In [ ]:
# Login to HuggingFace first
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
login(token=UserSecretsClient().get_secret("HF_TOKEN"))
print("Logged in!")

# Push LoRA adapter (small, ~100MB)
model.push_to_hub("covaga/eplan-assistant-v2-lora")
tokenizer.push_to_hub("covaga/eplan-assistant-v2-lora")
print("LoRA v2 adapter pushed!")

In [ ]:
from peft import PeftModel
import torch, glob
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# Free memory
try:
    del model, pipe
except NameError:
    pass
torch.cuda.empty_cache()

# Reload base model in full precision for merging
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
    low_cpu_mem_usage=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Find latest checkpoint
checkpoints = sorted(glob.glob("eplan-qwen-qlora-v2/checkpoint-*"))
lora_path = checkpoints[-1] if checkpoints else "eplan-qwen-qlora-v2"
print(f"Merging LoRA from: {lora_path}")

# Load and merge LoRA weights
merged_model = PeftModel.from_pretrained(base_model, lora_path)
merged_model = merged_model.merge_and_unload()

# Save merged model locally
merged_model.save_pretrained("eplan-assistant-v2-merged", safe_serialization=True, max_shard_size="2GB")
tokenizer.save_pretrained("eplan-assistant-v2-merged")
print("Merged v2 model saved locally!")

# Push merged model to Hub
merged_model.push_to_hub("covaga/eplan-assistant-v2-merged")
tokenizer.push_to_hub("covaga/eplan-assistant-v2-merged")
print("Merged v2 model pushed to HuggingFace!")

## 9. Convert to GGUF for Ollama

Convert the merged model to GGUF format with Q4_K_M quantization (~2GB).
This allows running the model locally with Ollama without needing Python or GPU.

In [ ]:
import subprocess, os

# Step 1: Install build tools
print("Installing build tools...")
subprocess.run(["apt-get", "update", "-qq"], check=True, capture_output=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "build-essential", "cmake"], check=True, capture_output=True)

# Step 2: Clone llama.cpp
print("Cloning llama.cpp...")
if not os.path.exists("/tmp/llama.cpp"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggerganov/llama.cpp.git", "/tmp/llama.cpp"], check=True)

# Step 3: Install llama.cpp Python deps
print("Installing Python deps...")
subprocess.run(["pip", "install", "-q", "gguf", "sentencepiece", "protobuf"], check=True, capture_output=True)

# Step 4: Convert to FP16 GGUF
print("Converting to GGUF FP16...")
subprocess.run([
    "python", "/tmp/llama.cpp/convert_hf_to_gguf.py",
    "eplan-assistant-v2-merged",
    "--outfile", "eplan-assistant-v2-f16.gguf",
    "--outtype", "f16",
], check=True)
print("FP16 GGUF created!")

# Step 5: Build quantize tool
print("Building quantize tool...")
os.makedirs("/tmp/llama.cpp/build", exist_ok=True)
subprocess.run([
    "cmake", "-B", "/tmp/llama.cpp/build", "-S", "/tmp/llama.cpp", "-DGGML_CUDA=OFF"
], check=True, capture_output=True)
subprocess.run([
    "cmake", "--build", "/tmp/llama.cpp/build", "--target", "llama-quantize", "-j", "4"
], check=True, capture_output=True)
print("Quantize tool built!")

# Step 6: Quantize to Q4_K_M
quantize_bin = "/tmp/llama.cpp/build/bin/llama-quantize"
print("Quantizing to Q4_K_M...")
subprocess.run([
    quantize_bin,
    "eplan-assistant-v2-f16.gguf",
    "eplan-assistant-v2-q4_k_m.gguf",
    "Q4_K_M",
], check=True)

# Check sizes
f16_size = os.path.getsize("eplan-assistant-v2-f16.gguf") / (1024**3)
q4_size = os.path.getsize("eplan-assistant-v2-q4_k_m.gguf") / (1024**3)
print(f"\nF16:    {f16_size:.2f} GB")
print(f"Q4_K_M: {q4_size:.2f} GB")
print(f"Compression: {f16_size/q4_size:.1f}x")

In [ ]:
# Upload GGUF to HuggingFace
from huggingface_hub import HfApi

api = HfApi()
repo_id = "covaga/eplan-assistant-v2-gguf"
api.create_repo(repo_id, exist_ok=True)

print("Uploading Q4_K_M GGUF...")
api.upload_file(
    path_or_fileobj="eplan-assistant-v2-q4_k_m.gguf",
    path_in_repo="eplan-assistant-v2-q4_k_m.gguf",
    repo_id=repo_id,
    commit_message="Add Q4_K_M GGUF (quantized from v2 merged model)",
)
print(f"Done! Download with:")
print(f"  hf download {repo_id} eplan-assistant-v2-q4_k_m.gguf")
print(f"\nOllama setup:")
print(f'  echo "FROM ./eplan-assistant-v2-q4_k_m.gguf" > Modelfile')
print(f"  ollama create eplan-assistant -f Modelfile")
print(f"  ollama run eplan-assistant")